# Marine Survey Photo & Coordinate Organizer

This notebook automates organizing marine transect photos and building an auditable Excel table formatted exactly like **`Example.xlsx`**.

### Workflow Overview
1. **Columns A - P (Survey & Coordinates)**: Extracted from GPS/waypoint CSV files in `coords/` (Location, Transect, Waypoint Point ID, Date, Time, Latitude, Longitude, and UTM projected coordinates).
   - **Multi-Zone UTM Support**: Automatically detects or allows specifying the UTM Zone (e.g. **Zone 48S / 49S** in Bangka Belitung, **Zone 51M** in Sulawesi).
2. **Column Q & R (Photo Previews)**: Image formula `=IMAGE(R{row})` pointing to the photo path / URL in Column R.
3. **Column S (Annotation/Interpretation)**: **Kept empty** for human marine annotator input.
4. **Columns T - AA (Image Metadata)**: Extracted directly from photo EXIF tags (File Size, Camera Model, DateOriginal, TimeOriginal, ShutterSpeed, Aperture, ISO, WhiteBalance).
5. **Photo Organization**: Copies and renames photos into structured folders (e.g., `output/organized_photos/D1T1/D1T1 (1).JPG`). Original source photos remain untouched.

## 1. Setup & Imports

In [ ]:
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from pathlib import Path
from photo_coordinate_organizer import (
    organize_photos,
    load_coordinates,
    extract_photo_metadata,
    latlon_to_utm
)

# Configure paths
PHOTOS_DIR = Path('Photos')
COORDS_DIR = Path('coords')
OUTPUT_DIR = Path('output')
TEAM_ID = 'A'

print('Setup complete.')

## 2. Inspect Coordinates in `coords/`
Let's inspect the waypoints and coordinates loaded from the CSV files in `coords/`:

In [ ]:
coords = load_coordinates(COORDS_DIR)
print(f'Loaded {len(coords)} coordinate points from {COORDS_DIR}.')

# Display first 5 points
sample_data = [
    {
        'Day': pt.day,
        'Point ID': pt.point_id,
        'Title': pt.title,
        'Latitude': pt.latitude,
        'Longitude': pt.longitude,
        'UTM Zone': f'{pt.utm_zone}{pt.utm_band}',
        'UTM Easting (X)': pt.utm_x,
        'UTM Northing (Y)': pt.utm_y,
        'Survey Date': pt.survey_date,
        'Survey Time': pt.survey_time
    }
    for pt in list(coords.values())[:5]
]
pd.DataFrame(sample_data)

## 3. Inspect Photos & EXIF Metadata in `Photos/`
Let's inspect the photos in `Photos/` and extract their EXIF metadata:

In [ ]:
photo_files = sorted([p for p in PHOTOS_DIR.iterdir() if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])
photo_samples = []
for p in photo_files:
    meta = extract_photo_metadata(p)
    photo_samples.append({
        'Original Name': meta.original_filename,
        'Renamed Name': meta.renamed_filename,
        'Day': meta.day,
        'Transect': meta.transect,
        'Point ID': meta.point_id,
        'Camera Model': meta.camera_model,
        'ShutterSpeed': meta.shutter_speed,
        'Aperture': meta.aperture,
        'ISO': meta.iso,
        'DateOriginal': meta.date_original,
        'TimeOriginal': meta.time_original,
        'FileSize': meta.file_size_str,
    })

pd.DataFrame(photo_samples)

## 4. Run Organization & Excel Generation
Now we run `organize_photos` to:
1. Match photos with coordinates by Day and Point ID.
2. Project coordinates to UTM (auto-detecting Zone 51 in Sulawesi, Zone 48/49 in Bangka Belitung, or forced).
3. Copy/organize photos into `output/organized_photos/<Transect>/`.
4. Generate `output/Survey_Organized.xlsx` with the exact 27-column structure of `Example.xlsx`.

In [ ]:
records, excel_file = organize_photos(
    photos_dir=PHOTOS_DIR,
    coords_dir=COORDS_DIR,
    output_dir=OUTPUT_DIR,
    team=TEAM_ID,
    copy_mode='copy',       # 'copy' to duplicate, 'hardlink' to save disk space
    url_prefix='',          # optional URL prefix, e.g. 'https://drive.google.com/...'
    utm_zone='auto',        # 'auto' or specify 48 / 49 (Bangka Belitung) or 51 (Sulawesi)
    utm_format='MGRS',      # 'MGRS' (e.g. X_UTM48M, X_UTM51M) or 'HEMISPHERE' (e.g. X_UTM48S)
)

print(f'[SUCCESS] Organized {len(records)} photos into: {OUTPUT_DIR / "organized_photos"}')
print(f'[SUCCESS] Saved Excel report to: {excel_file.resolve()}')

## 5. Preview Generated Table
Let's inspect the resulting data table.
- Notice **Columns A - P** are extracted from the coordinate CSV files.
- Notice **Column S** (`Annotation/Interpretation`) remains empty for human annotators.
- Notice **Columns T - AA** are extracted from the photo metadata.

In [ ]:
df = pd.DataFrame(records)
print('--- Columns A to P (Coordinates from CSV) ---')
utm_cols = [c for c in df.columns if c.startswith('X_UTM') or c.startswith('Y_UTM')]
coord_cols = ['ID', 'Team', 'ID Transek', 'ID Titik', 'File name', 'Tanggal', 'Jam', 'Latitude', 'Longitude', 'Lokasi'] + utm_cols
display(df[coord_cols])

print('\n--- Columns Q to AA (Photo Links & Image Metadata) ---')
# Column S is empty
display(df[['ID Titik', 'Photo', 'URL', 'Annotation/Interpretation', 'FileSize', 'Model', 'DateOriginal', 'TimeOriginal', 'ShutterSpeed', 'Aperture', 'ISO', 'WhiteBalance']])